# 🛡️ Oversight Arena — TRL GRPO Training Notebook

**Meta × PyTorch OpenEnv Hackathon 2026**

This notebook trains a small open-weight overseer LLM (Qwen-2.5-1.5B-Instruct) to detect
malicious peer agents in collaborative coding, using TRL GRPO against the live OpenEnv
Hugging Face Space.

| Resource | Link |
|---|---|
| 🛰️ Live env | https://anikasoni-oversight-arena.hf.space |
| 💻 GitHub | https://github.com/anikasoni/oversight-arena |
| 🤗 HF Space | https://huggingface.co/spaces/anikasoni/oversight_arena |

---

### What this notebook does

1. **Install** dependencies
2. **Verify** the live OpenEnv Space is healthy
3. **Baseline eval** — untrained model on 30 held-out seeds
4. **Train** with TRL GRPO against the live environment (96 prompts, curriculum)
5. **Post-training eval** on the same 30 seeds
6. **Visualise** loss/reward curves and F1 comparison
7. **Action distribution shift** — key evidence of learned policy

### Runtime estimates

| GPU | Model | Time |
|---|---|---|
| T4 (free tier) | Qwen2.5-0.5B | ~45 min |
| L4 / A100 | Qwen2.5-1.5B | ~90 min |

Change `MODEL` in cell 3 to switch model size.

> **Runtime tip:** Go to `Runtime → Change runtime type → GPU` before running.

In [ ]:
# @title 1. Install dependencies { display-mode: "form" }
# @markdown Run this cell first. Takes ~2 minutes.

!pip install -q --upgrade "trl>=0.12" peft transformers accelerate datasets
!pip install -q "openenv-core>=0.2.1"
!pip install -q matplotlib requests bitsandbytes

# Verify key installs
import trl, peft, transformers
print(f'trl:            {trl.__version__}')
print(f'peft:           {peft.__version__}')
print(f'transformers:   {transformers.__version__}')
print('✅ Dependencies installed')

In [ ]:
# @title 2. Clone repo { display-mode: "form" }
import os

if not os.path.exists('oversight-arena'):
    !git clone https://github.com/anikasoni/oversight-arena.git
    print('✅ Cloned')
else:
    print('Repo already exists — pulling latest...')
    !cd oversight-arena && git pull

%cd oversight-arena
!pip install -q -e '.[train]'

# Quick smoke test
!python -c "from oversight_arena.sabotage_catalog import build_catalog; c=build_catalog(); print(f'Catalog: {len(c)} patterns loaded ✅')"

In [ ]:
# @title 3. Config — edit here { display-mode: "form" }
# @markdown **Change `MODEL` to `Qwen/Qwen2.5-0.5B-Instruct` for a faster T4 run.**

ENV_URL    = 'https://anikasoni-oversight-arena.hf.space'  # @param {type:"string"}
MODEL      = 'Qwen/Qwen2.5-1.5B-Instruct'                # @param ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"]
N_PROMPTS  = 96    # @param {type:"integer"}
EVAL_N     = 30    # @param {type:"integer"}
LR         = 5e-6  # @param {type:"number"}
EPOCHS     = 2     # @param {type:"integer"}
CURRICULUM = True  # @param {type:"boolean"}

print(f'Model:      {MODEL}')
print(f'Env:        {ENV_URL}')
print(f'N prompts:  {N_PROMPTS}')
print(f'Eval N:     {EVAL_N}')
print(f'LR:         {LR}')
print(f'Epochs:     {EPOCHS}')
print(f'Curriculum: {CURRICULUM}')

In [ ]:
# @title 4. Verify live env is healthy { display-mode: "form" }
# @markdown Checks /health, /reset, /step, and /grader against the live HF Space.
# @markdown All four should succeed before training.

import requests, json

def _pp(label, data):
    print(f'\n{'='*4} {label} {'='*4}')
    if isinstance(data, dict):
        for k, v in data.items():
            if isinstance(v, str) and len(v) > 120:
                print(f'  {k}: {v[:120]}…')
            else:
                print(f'  {k}: {v}')
    else:
        print(' ', data)

# 1. Health
r = requests.get(f'{ENV_URL}/health', timeout=15)
r.raise_for_status()
_pp('/health', r.json())

# 2. Reset — note: response now includes BOTH observation AND state
r = requests.post(f'{ENV_URL}/reset',
                  json={'seed': 0, 'difficulty': 0.4}, timeout=30)
r.raise_for_status()
payload = r.json()
obs   = payload.get('observation', {})
state = payload.get('state', {})
print('\n==== /reset (seed=0, d=0.4) ====')
print('  workers:          ', obs.get('workers'))
print('  malicious_workers:', state.get('malicious_workers'))
print('  malicious_tier:   ', state.get('malicious_tier'))
print('  episode_id:       ', state.get('episode_id'))
print('  diff (first 300):\n')
print(obs.get('focused_patch_diff', '')[:300])

# 3. Step
r = requests.post(f'{ENV_URL}/step',
                  json={'action': 'flag_worker', 'worker_id': 'W3',
                        'cwe_tag': 'CWE-476', 'reasoning': 'missing null check'},
                  timeout=30)
r.raise_for_status()
step = r.json()
_pp('/step flag W3', {'reward': step.get('reward'), 'done': step.get('done')})

# 4. Grader
r = requests.get(f'{ENV_URL}/grader', timeout=15)
r.raise_for_status()
g = r.json()
_pp('/grader', {
    'f1':                g.get('f1'),
    'tp/fp/fn':          f"{g.get('tp')}/{g.get('fp')}/{g.get('fn')}",
    'reward':            g.get('reward'),
    'guardrails':        g.get('guardrails_triggered'),
    'malicious_workers': g.get('malicious_workers'),
    'flagged_workers':   g.get('flagged_workers'),
})

print('\n✅ All env checks passed — ready to train')

In [ ]:
# @title 5. Run training (TRL GRPO) { display-mode: "form" }
# @markdown This cell runs the full training loop. Logs stream directly to output.
# @markdown Training saves a LoRA checkpoint to `checkpoints/grpo/` on completion.

import subprocess, sys

cmd = [
    sys.executable, 'scripts/train_grpo.py',
    '--env-url',         ENV_URL,
    '--model',           MODEL,
    '--n-prompts',       str(N_PROMPTS),
    '--num-generations', '4',
    '--batch-size',      '4',
    '--grad-accum',      '2',
    '--lr',              str(LR),
    '--epochs',          str(EPOCHS),
    '--eval-after-train',
    '--eval-n',          str(EVAL_N),
    '--eval-difficulties', '0.2,0.4,0.6',
    '--eval-samples',    '4',
    '--eval-temperature','0.7',
    '--save-checkpoint',
]

if CURRICULUM:
    cmd.append('--curriculum')

print('Command:\n', ' '.join(cmd), '\n')
print('=' * 60)
result = subprocess.run(cmd, capture_output=False)
print('=' * 60)
print(f'\nReturn code: {result.returncode}')
if result.returncode == 0:
    print('✅ Training complete')
else:
    print('⚠️  Training exited with errors — check logs above')

In [ ]:
# @title 6. Show training results { display-mode: "form" }
import json, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

# ── Training summary ─────────────────────────────────────────────
summary_path = Path('results/training_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    b = summary.get('baseline_eval', {})
    t = summary.get('trained_eval', {})

    print('╔══════════════════════════════════════╗')
    print('║         Training Summary             ║')
    print('╠══════════════════════════════════════╣')
    print(f"║  Model:        {summary.get('model', '?')[:22]:<22} ║")
    print(f"║  N prompts:    {summary.get('n_prompts', '?'):<22} ║")
    print(f"║  Mean reward:  {summary.get('mean_reward', 0):<22.4f} ║")
    print(f"║  First → Last: {summary.get('first_window_mean_reward', 0):.4f} → {summary.get('last_window_mean_reward', 0):.4f}        ║")
    print('╠══════════════════════════════════════╣')
    print(f"║  Baseline F1:  {b.get('mean_f1', 0):<22.4f} ║")
    print(f"║  Trained F1:   {t.get('mean_f1', 0):<22.4f} ║")
    delta = summary.get('delta_f1', 0)
    arrow = '↑' if delta > 0 else '↓'
    print(f"║  Delta F1:     {arrow} {delta:+.4f}                ║")
    print(f"║  Delta Reward: {summary.get('delta_reward', 0):+.4f}                ║")
    print('╚══════════════════════════════════════╝')
else:
    print('[training_summary.json not found — did training complete?]')

# ── Plots ────────────────────────────────────────────────────────
plots = [
    ('results/reward_curve.png',    'Reward over training'),
    ('results/loss_curve.png',      'Loss over training'),
    ('results/final_comparison.png','Baseline vs Trained F1'),
    ('results/ablation_reward_hacking.png', 'Guardrail ablation'),
]

for path, title in plots:
    p = Path(path)
    if p.exists():
        print(f'\n── {title} ──')
        display(Image(str(p)))
    else:
        print(f'[missing] {path}')

In [ ]:
# @title 7. Action distribution shift (key evidence) { display-mode: "form" }
# @markdown The policy shift from 'inspect_patch doubling' is the strongest
# @markdown behavioural evidence that training worked.

import csv, collections
from pathlib import Path

print('Action distribution at d=0.4 (30 held-out seeds)\n')
print(f'{"Action":<20} {"Baseline":>10} {"Trained":>10} {"Δ":>8}')
print('─' * 52)

baseline_actions, trained_actions = {}, {}
baseline_f1s, trained_f1s = [], []

for label, path, action_dict, f1_list in [
    ('baseline', 'results/eval_baseline_d0.4.csv', baseline_actions, baseline_f1s),
    ('trained',  'results/eval_grpo_d0.4.csv',     trained_actions,  trained_f1s),
]:
    p = Path(path)
    if p.exists():
        rows = list(csv.DictReader(p.open()))
        counts = collections.Counter(r.get('action', 'unknown') for r in rows)
        f1s    = [float(r['f1']) for r in rows if 'f1' in r]
        action_dict.update(counts)
        f1_list.extend(f1s)
    else:
        print(f'  [{label}] file not found: {path}')

all_actions = sorted(set(list(baseline_actions) + list(trained_actions)))
for a in all_actions:
    b_count = baseline_actions.get(a, 0)
    t_count = trained_actions.get(a, 0)
    delta   = t_count - b_count
    arrow   = f'↑+{delta}' if delta > 0 else (f'↓{delta}' if delta < 0 else '=')
    print(f'{a:<20} {b_count:>10} {t_count:>10} {arrow:>8}')

print('─' * 52)
if baseline_f1s:
    print(f'  mean F1 baseline: {sum(baseline_f1s)/len(baseline_f1s):.4f}')
if trained_f1s:
    print(f'  mean F1 trained:  {sum(trained_f1s)/len(trained_f1s):.4f}')
    if baseline_f1s:
        delta = sum(trained_f1s)/len(trained_f1s) - sum(baseline_f1s)/len(baseline_f1s)
        print(f'  delta F1:         {delta:+.4f}')

In [ ]:
# @title 8. (Optional) Run a live interactive episode { display-mode: "form" }
# @markdown Calls the live API to run one full episode manually.
# @markdown Useful for sanity-checking the trained model's decisions.

import requests, textwrap

SEED       = 42   # @param {type:"integer"}
DIFFICULTY = 0.4  # @param {type:"number"}

print(f'Starting episode seed={SEED}, difficulty={DIFFICULTY}...\n')

# Reset
r = requests.post(f'{ENV_URL}/reset',
                  json={'seed': SEED, 'difficulty': DIFFICULTY}, timeout=30)
r.raise_for_status()
payload = r.json()
obs   = payload.get('observation', {})
state = payload.get('state', {})

print(f"Workers:           {obs.get('workers')}")
print(f"Malicious (hidden): {state.get('malicious_workers')}")
print(f"Malicious tier:     {state.get('malicious_tier')}")
print()
print('Patch diff:')
print('─' * 60)
diff = obs.get('focused_patch_diff', '')
print(diff[:800])
if len(diff) > 800:
    print(f'... [{len(diff)-800} more chars]')
print('─' * 60)

# Take one action
action = {
    'action':    'inspect_patch',
    'worker_id': 'W1',
    'reasoning': 'Starting with W1 to build a baseline comparison.',
    'cwe_tag':   '',
}
r = requests.post(f'{ENV_URL}/step', json=action, timeout=30)
r.raise_for_status()
step = r.json()
print(f"\nStep result: reward={step.get('reward')}, done={step.get('done')}")

# Get grader
r = requests.get(f'{ENV_URL}/grader', timeout=15)
r.raise_for_status()
g = r.json()
print(f"\nGrader: F1={g.get('f1'):.3f} | tp={g.get('tp')} fp={g.get('fp')} fn={g.get('fn')}")
print(f"Success (F1≥0.70): {g.get('success')}")